# CS781 Phase 6 - final submission list (local, CPU, standard library only)
Reads every `bundle_p6_*.zip` in `BUNDLE_DIR` (notebooks 1-10; rerun after adding new bundles) and writes ranked,
ready-to-upload submission files to `OUT_DIR`.

**Candidates.** Every run x every summary length scored on the held-out val split (200 / 275 / 350 / 425 words /
untruncated). The test summaries in the bundles are untruncated, so each length is a different, valid submission.
**Ranking** (all by val BERTScore F1, the primary metric):
0. files already uploaded (`ALREADY_SUBMITTED`) keep their names and numbers; new files are numbered after them;
1. each run at its own best length - every experiment gets on the leaderboard (the brief needs all techniques and models);
2. the remaining run x length variants, best first, skipping variants within 20 words of one already listed for that run;
3. the extractive baseline (`X00_extractive_baseline.zip`, if you copy it into `BUNDLE_DIR`) last, as a reference.
Capped at `MAX_FILES` (30) minus `RESERVE` (slots kept for notebooks still running). Files already uploaded (`ALREADY_SUBMITTED`) are kept in the plan but marked, so after adding
bundles 7-10 the list shows what is new.

In [1]:
BUNDLE_DIR = r"D:\IIT Kanpur\Large Language Models\Capstone\Code\Phase-6\bundles"   # bundle_p6_1.zip ... bundle_p6_10.zip
OUT_DIR = None                            # default: <BUNDLE_DIR>/final_submissions
MAX_FILES = 40                            # 30 already uploaded today + the 10 left under the 40/day limit
RESERVE = 0                               # all notebooks are in
# Files already uploaded to Codabench keep their names and numbers; new files are numbered after them (31-40).
# Optional: copy LEN_<run>.zip files from the length notebook into BUNDLE_DIR; they are ranked right after the new runs.
ALREADY_SUBMITTED = [                     # the 30 files actually uploaded (Codabench list, 3 Nov)
    "01_D08_full_chunk_425w", "02_B09_full_chunk_425w", "03_A12_full_sled_275w", "04_D04_lorma_extract_275w",
    "05_D10_adalora_extract_275w", "06_D11_adapter_extract_275w", "07_C13_lorma_extract_275w", "08_E02_kd_extract_275w",
    "09_B07_lorma_extract_275w", "10_D01_full_extract_275w", "11_B06_full_extract_275w", "12_E03_full_extract_275w",
    "13_D05_full_lead_200w", "14_D08_full_chunk_350w", "15_D08_full_chunk_allw", "16_D08_full_chunk_275w",
    "17_B09_full_chunk_allw", "18_B09_full_chunk_350w", "19_B09_full_chunk_275w", "20_D08_full_chunk_200w",
    "21_A12_full_sled_350w", "22_B09_full_chunk_200w", "23_A12_full_sled_425w", "24_D04_lorma_extract_200w",
    "25_D04_lorma_extract_350w", "26_A12_full_sled_allw", "27_A12_full_sled_200w", "28_D10_adalora_extract_350w",
    "29_B07_lorma_extract_200w", "30_D10_adalora_extract_200w",
]

In [2]:
import csv, glob, io, json, os, zipfile
OUT_DIR = OUT_DIR or os.path.join(BUNDLE_DIR, "final_submissions")
os.makedirs(OUT_DIR, exist_ok=True)

def cut_words(t, n):                      # identical to summ_data.cut_words (how val was scored)
    w = t.split()
    return " ".join(w[:n]) if n else " ".join(w)

runs = {}
bundles = sorted(glob.glob(os.path.join(BUNDLE_DIR, "bundle_p6_*.zip")))
assert bundles, f"no bundle_p6_*.zip in {BUNDLE_DIR}"
for b in bundles:
    z = zipfile.ZipFile(b)
    names = set(z.namelist())
    for n in sorted(names):
        if not n.endswith("/summary.json"):
            continue
        tag = n.split("/")[0]
        if f"{tag}/test_gen.json" not in names:
            print(f"skip {tag}: no test summaries"); continue
        s = json.load(z.open(n))
        test = json.load(z.open(f"{tag}/test_gen.json"))
        runs[tag] = dict(summary=s, test=test, bundle=os.path.basename(b))
print(f"{len(bundles)} bundles, {len(runs)} runs: {', '.join(sorted(runs))}")

# all test files must cover the same documents
ref_ids = None
for tag, r in runs.items():
    ids = [x["id"] for x in r["test"]]
    assert len(ids) == len(set(ids)), f"{tag}: duplicate test ids"
    assert all(x["summary"].strip() for x in r["test"]), f"{tag}: empty summary"
    if ref_ids is None:
        ref_ids = sorted(ids)
    assert sorted(ids) == ref_ids, f"{tag}: different test documents"
print(f"test documents: {len(ref_ids)} (identical in every run)")

10 bundles, 20 runs: A12_full_sled, A20_full_sled8k, B06_full_extract, B07_lorma_extract, B09_full_chunk, B15_lorma_chunk, B19_lorma_stage2, C13_lorma_extract, C17_lorma_chunk, D01_full_extract, D04_lorma_extract, D05_full_lead, D08_full_chunk, D10_adalora_extract, D11_adapter_extract, D14_lorma_chunk, D18_lorma_stage2, E02_kd_extract, E03_full_extract, E16_kd_chunk
test documents: 100 (identical in every run)


In [3]:
def tech(s):
    a = s["args"]
    m = a["method"]
    t = {"full": "Full FT", "lorma": "LoRMA", "adalora": "AdaLoRA", "adapter": "Houlsby adapters", "lora": "LoRA",
         "qlora": "QLoRA"}.get(m, m)
    if a.get("teacher"):
        t = "KD (" + os.path.basename(a["teacher"]).split("_")[0] + " -> student)"
    return t

cands = []
for tag, r in runs.items():
    s = r["summary"]
    for row in s["val_by_length"]:
        if "bertscore_f1" not in row:
            continue
        cands.append(dict(tag=tag, words=row["words"], bs=row["bertscore_f1"], r1=row["rouge1"], rl=row["rougeL"],
                          mean_words=row["mean_words"], model=tag[0], tech=tech(s), input=s["args"]["input"],
                          best=(row["words"] == s["chosen"]["words"])))
CAP = MAX_FILES - RESERVE
key = lambda c: f"{c['tag']}_{c['words'] or 'all'}w"
done_name = {n.split("_", 1)[1]: n for n in ALREADY_SUBMITTED}          # "D08_full_chunk_425w" -> "01_D08_full_chunk_425w"
for c in cands:
    c["name"] = done_name.get(key(c))
plan, used = [], {}
def take(c, status):
    plan.append(c); used.setdefault(c["tag"], []).append(c["mean_words"]); c["status"] = status
# 1. everything already uploaded, in its original order (always kept, counts toward the cap)
for c in sorted([c for c in cands if c["name"]], key=lambda c: c["name"]):
    take(c, "kept: already submitted")
n_forced = len(plan)
missing = sorted(set(done_name) - {key(c) for c in cands if c["name"]})
if missing:
    print("WARNING - listed as submitted but not found in the bundles:", missing)
# 2. every run not yet on the leaderboard, at its own best length
best = sorted([c for c in cands if c["best"] and not c["name"]], key=lambda c: -c["bs"])
for c in best:
    if c["tag"] in used:                          # this run is already on the leaderboard at some length
        c["status"] = "skipped: run already submitted at another length"; continue
    take(c, "kept: best length of its run")
# 3. other lengths, best first, until the cap
rest = sorted([c for c in cands if "status" not in c], key=lambda c: -c["bs"])
for c in rest:
    near = [w for w in used.get(c["tag"], []) if abs(c["mean_words"] - w) < 20]
    if near:
        c["status"] = f"skipped: within 20 words of a kept length of this run ({c['mean_words']:.0f} vs {near[0]:.0f} words)"
        continue
    if len(plan) >= CAP:
        c["status"] = f"skipped: over the cap of {CAP} files"
        continue
    take(c, "kept: other length")
for c in best:                                    # a best length that did not fit (only if CAP < number of runs)
    c.setdefault("status", "skipped")
# length-rule files from cs781-p6-length-rule (only written there when their cross-fitted val gain was positive);
# they go before the plain extra lengths: insert them right after the last best-length entry
lens = sorted(glob.glob(os.path.join(BUNDLE_DIR, "LEN_*.zip")))
if lens:
    pos = max([i for i, c in enumerate(plan) if c["status"].startswith("kept: best")] + [n_forced - 1]) + 1
    for f in lens:
        tag = os.path.basename(f)[4:-4]
        plan.insert(pos, dict(tag=tag, words="len", bs=float("nan"), r1=float("nan"), rl=float("nan"), mean_words=0,
                              model=tag[0], tech="length per judgment", input="chunk", best=False, name=None,
                              status="kept: length chosen per judgment (cross-fitted gain > 0)", len_file=f))
        pos += 1
    while len(plan) > CAP and plan[-1]["status"] == "kept: other length":
        c = plan.pop(); c["status"] = f"skipped: over the cap of {CAP} files (made room for LEN files)"
x00 = os.path.join(BUNDLE_DIR, "X00_extractive_baseline.zip")
if os.path.exists(x00) and len(plan) < CAP:
    plan.append(dict(tag="X00_extractive_baseline", words=400, bs=float("nan"), r1=float("nan"), rl=float("nan"),
                     mean_words=400, model="-", tech="extractive (no model)", input="extract", best=True, name=None,
                     status="kept: reference"))

In [4]:
for f in glob.glob(os.path.join(OUT_DIR, "*.zip")):   # files from an earlier run of this notebook
    os.remove(f)
rows = []
n_done = sum(1 for c in plan if c.get("name"))
nxt = n_done
for c in plan:
    if c.get("name"):
        name = c["name"]
    else:
        nxt += 1
        name = f"{nxt:02d}_{c['tag']}_{c['words'] or 'all'}w" if c["tag"] != "X00_extractive_baseline" else f"{nxt:02d}_X00_extractive_baseline"
    path = os.path.join(OUT_DIR, name + ".zip")
    if c.get("len_file"):
        name = f"{nxt:02d}_LEN_{c['tag']}"; path = os.path.join(OUT_DIR, name + ".zip")
        with zipfile.ZipFile(c["len_file"]) as zi, zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as zo:
            zo.writestr("prediction.json", zi.read("prediction.json"))
    elif c["tag"] == "X00_extractive_baseline":
        with zipfile.ZipFile(x00) as zi, zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as zo:
            zo.writestr("prediction.json", zi.read("prediction.json"))
    else:
        pred = [{"id": x["id"], "summary": cut_words(x["summary"], c["words"]) or "."} for x in runs[c["tag"]]["test"]]
        with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as zo:
            zo.writestr("prediction.json", json.dumps(pred, ensure_ascii=False))
    c["file"] = name
    rows.append(dict(rank=int(name[:2]), file=name + ".zip", run=c["tag"], model=c["model"].upper(), technique=c["tech"],
                     input=c["input"], words=c["words"] or "all", val_bertscore=round(c["bs"], 4) if c["bs"] == c["bs"] else float("nan"),
                     val_rouge1=round(c["r1"], 4), val_rougeL=round(c["rl"], 4),
                     why=c["status"].replace("kept: ", ""),
                     status="already submitted" if c.get("name") else "TO SUBMIT"))
rows.sort(key=lambda r: r["rank"])
# no new file may be identical to another file (identical submissions do not count on Codabench)
import hashlib
seen = {}
for r in rows:
    with zipfile.ZipFile(os.path.join(OUT_DIR, r["file"])) as z:
        h = hashlib.md5(json.dumps(sorted((x["id"], x["summary"]) for x in json.loads(z.read("prediction.json")))).encode()).hexdigest()
    assert h not in seen, f"{r['file']} is identical to {seen[h]} - remove one before uploading"
    seen[h] = r["file"]
print("duplicate check OK: all files have different content")
with open(os.path.join(OUT_DIR, "submission_plan.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0]))
    w.writeheader(); w.writerows(rows)
print(f"{len(rows)} files in {OUT_DIR}  (+ submission_plan.csv): {n_done} already submitted, "
      f"{len(rows) - n_done} TO SUBMIT (upload in this order)\n")
print(f"{'#':>2} {'file':42s} {'model':5s} {'technique':24s} {'input':8s} {'words':>5s} {'BERTScore':>9s} {'R-1':>6s} {'R-L':>6s}  status")
for r in rows:
    print(f"{r['rank']:2d} {r['file']:42s} {r['model']:5s} {r['technique'][:24]:24s} {r['input'][:8]:8s} {str(r['words']):>5s} "
          f"{r['val_bertscore']:9.4f} {r['val_rouge1']:6.4f} {r['val_rougeL']:6.4f}  {r['status']}")
# self-check: every file has one summary per test document
for r in rows:
    with zipfile.ZipFile(os.path.join(OUT_DIR, r["file"])) as z:
        p = json.loads(z.read("prediction.json"))
    assert sorted(x["id"] for x in p) == ref_ids and all(x["summary"].strip() for x in p), r["file"]
print("\nall files checked: one non-empty summary per test document, list format [{id, summary}]")

duplicate check OK: all files have different content
40 files in D:\IIT Kanpur\Large Language Models\Capstone\Code\Phase-6\bundles\final_submissions  (+ submission_plan.csv): 30 already submitted, 10 TO SUBMIT (upload in this order)

 # file                                       model technique                input    words BERTScore    R-1    R-L  status
 1 01_D08_full_chunk_425w.zip                 D     Full FT                  chunk      425    0.8513 0.4919 0.2702  already submitted
 2 02_B09_full_chunk_425w.zip                 B     Full FT                  chunk      425    0.8498 0.4900 0.2740  already submitted
 3 03_A12_full_sled_275w.zip                  A     Full FT                  sled       275    0.8467 0.3950 0.2241  already submitted
 4 04_D04_lorma_extract_275w.zip              D     LoRMA                    extract    275    0.8460 0.3958 0.2231  already submitted
 5 05_D10_adalora_extract_275w.zip            D     AdaLoRA                  extract    275    0.8447 

In [5]:
# EVERY candidate (run x length) and what happened to it
kept = {(c["tag"], c["words"]): int(c["file"][:2]) for c in plan}
print(f"candidates: {len(cands)} (= {len(runs)} runs x lengths) | kept: {len(plan)} | "
      f"skipped as near-duplicates: {sum('within 20' in c['status'] for c in cands)} | "
      f"skipped over the cap: {sum('over the cap' in c['status'] for c in cands)} | "
      f"to submit now: {len(plan) - n_done}\n")
print(f"{'run':22s} {'words':>5s} {'mean':>5s} {'BERTScore':>9s} {'R-1':>6s}  decision")
for c in sorted(cands, key=lambda c: (c["tag"], c["words"] or 10**6)):
    r = kept.get((c["tag"], c["words"]))
    print(f"{c['tag']:22s} {str(c['words'] or 'all'):>5s} {c['mean_words']:5.0f} {c['bs']:9.4f} {c['r1']:6.4f}  "
          + (f"#{r:02d} " if r else "    ") + c["status"])
with open(os.path.join(OUT_DIR, "all_candidates.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["run", "words", "mean_words", "val_bertscore", "val_rouge1", "val_rougeL", "rank", "decision"])
    for c in sorted(cands, key=lambda c: (c["tag"], c["words"] or 10**6)):
        w.writerow([c["tag"], c["words"] or "all", round(c["mean_words"]), round(c["bs"], 4), round(c["r1"], 4),
                    round(c["rl"], 4), kept.get((c["tag"], c["words"]), ""), c["status"]])
print(f"\nfull list also saved as {os.path.join(OUT_DIR, 'all_candidates.csv')}")

candidates: 100 (= 20 runs x lengths) | kept: 40 | skipped as near-duplicates: 0 | skipped over the cap: 60 | to submit now: 10

run                    words  mean BERTScore    R-1  decision
A12_full_sled            200   200    0.8443 0.3356  #27 kept: already submitted
A12_full_sled            275   275    0.8467 0.3950  #03 kept: already submitted
A12_full_sled            350   345    0.8465 0.4300  #21 kept: already submitted
A12_full_sled            425   410    0.8450 0.4503  #23 kept: already submitted
A12_full_sled            all   496    0.8445 0.4641  #26 kept: already submitted
A20_full_sled8k          200   200    0.8438 0.3374      skipped: over the cap of 40 files
A20_full_sled8k          275   275    0.8463 0.3981      skipped: over the cap of 40 files
A20_full_sled8k          350   347    0.8468 0.4368  #35 kept: best length of its run
A20_full_sled8k          425   414    0.8460 0.4606      skipped: over the cap of 40 files
A20_full_sled8k          all   506    0.8457 